In [2]:
import time
from typing import Final
script_start: Final = time.perf_counter() # Do not remove or change this value

## 1. Setup

Edit this cell to configure your environment (paths, seeds, device selection, etc.).


In [3]:
# Standard library utilities for filesystem access, archive handling (ZIP/TAR),
# text parsing, JSON metadata, and basic timing/math utilities.
# These are typically used when loading datasets packaged as archives
# and normalising them into a common on-disk structure.
import os, io, re, zipfile, tarfile, json, math, time
from pathlib import Path
from collections import defaultdict

# Core numerical and image-processing libraries.
# NumPy underpins almost all geometric and numerical operations,
# while PIL is used for loading, saving, and manipulating fragment images.
import numpy as np
from PIL import Image

# PyTorch core library and neural network components.
# These imports support defining models that predict fragment pose
# (e.g. x, y position and rotation) and training them end-to-end.
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dataset and DataLoader abstractions.
# These are used to stream large fragment collections efficiently,
# either as standard indexed datasets or iterable datasets for
# very large or on-the-fly generated puzzles.
from torch.utils.data import IterableDataset, DataLoader
from torch.utils.data import Dataset, DataLoader

# Lightweight data containers and typing utilities.
# Dataclasses are often used to store structured metadata
# (e.g. fragment attributes, puzzle configuration),
# while typing improves code clarity in complex pipelines.
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

# Image preprocessing and augmentation utilities.
# These are typically applied to fragment images before passing
# them through a neural network (normalisation, resizing, cropping).
from torchvision import transforms

# Python-side randomness utilities.
# Used for lightweight sampling or shuffling when NumPy or PyTorch
# RNGs are not required.
import random

# Progress bar utility.
# Commonly wrapped around training loops or dataset preprocessing
# to give real-time feedback on long-running operations.
from tqdm import tqdm

# OpenCV computer vision library.
# Frequently used for contour extraction, edge detection,
# morphological operations, and geometric reasoning on fragments.
import cv2

# Utility for visualising batches of images as a single grid.
# Useful for quickly inspecting fragment sets or model inputs/outputs.
from torchvision.utils import make_grid

# Duplicate imports are intentionally retained so that this cell
# can be copy-pasted or partially reused without dependency issues.
import numpy as np
from PIL import Image

# Visualisation library for qualitative inspection and debugging.
# Common uses include plotting fragment crops, contours,
# intermediate feature maps, and reconstructed layouts.
import matplotlib.pyplot as plt

In [4]:
# Paths and dataset locations
DATA_ROOT = Path("../dataset")          # Root folder containing all datasets
COCOTILES_ZIP = DATA_ROOT / "CocoTiles.zip"
DAFNE_ZIP     = DATA_ROOT / "Dafne.zip"
OUTPUT_MODEL = "example_username-model.pth"

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Dataset and augmentation settings
AUGMENTATION = "None"                 # Options: "None", "Simple", "Moderate", "Hard"
FRAGMENT_FIXED_SIZE = (24, 24)        # (width, height) in pixels
NORMALIZE_RGB = True

# Training and model configuration
EPOCHS = 10
LEARNING_RATE=3e-4
NUMBER_WORKERS=0
BATCH_SIZE=4

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
print("device:", device)

DEVICE = device


DATASET_ZIP=COCOTILES_ZIP
DATASET_MODE= 'None' # Note can be simple ...

# Configuration summary (sanity check)
print("DATA_ROOT:      ", DATA_ROOT)
print("COCOTILES_ZIP:  ", COCOTILES_ZIP)
print("DAFNE_ZIP:      ", DAFNE_ZIP)
print("AUGMENTATION:   ", AUGMENTATION)
print("EPOCHS:         ", EPOCHS)
print("FRAGMENT_SIZE:  ", FRAGMENT_FIXED_SIZE)
print("NORMALIZE_RGB:  ", NORMALIZE_RGB)
print("DEVICE:         ", DEVICE)
print("================")
print("RUNNING ON      ",DATASET_ZIP)


device: cuda
DATA_ROOT:       ..\dataset
COCOTILES_ZIP:   ..\dataset\CocoTiles.zip
DAFNE_ZIP:       ..\dataset\Dafne.zip
AUGMENTATION:    None
EPOCHS:          10
FRAGMENT_SIZE:   (24, 24)
NORMALIZE_RGB:   True
DEVICE:          cuda
RUNNING ON       ..\dataset\CocoTiles.zip


## 2. Data Loading (Optional)

If you are training any models, you can add your **data loading utilities** here.

You may choose to:
- Load MS-COCO tile patches as training samples.
- Load DAFNE fragment crops or descriptors.
- Build small datasets for training lightweight heads on top of **frozen** pretrained backbones.

Remember:
- You must **not** use any external labelled data (COCO labels, segmentation maps, etc.).
- Only frozen model-zoo backbones may be used as feature extractors.


In [5]:
from UnifiedPuzzleSetZipDataset import make_unified_puzzle_dataloader_zip, batch_fragments_from_collate

## 3. Model Definition (Optional)

Define any **trainable components** of your system here (e.g., small MLPs, attention layers, or other heads on top of frozen pretrained features).

All **trainable parameters at inference** must sum to **≤ 15M**.

In [6]:
# ----------------------------
# Global pooling helpers (mask-aware)
# ----------------------------
def masked_mean(x: torch.Tensor, valid_mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    x: [B, N, D]
    valid_mask: [B, N] bool, True where VALID (not pad)
    returns: [B, D]
    """
    m = valid_mask.unsqueeze(-1).type_as(x)      # [B,N,1]
    s = (x * m).sum(dim=1)                       # [B,D]
    denom = m.sum(dim=1).clamp_min(eps)          # [B,1]
    return s / denom

def masked_max(x: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:
    """
    x: [B, N, D]
    valid_mask: [B, N] bool, True where VALID (not pad)
    returns: [B, D]
    """
    # set invalid positions to -inf so they never win the max
    neg_inf = torch.finfo(x.dtype).min
    x2 = x.masked_fill(~valid_mask.unsqueeze(-1), neg_inf)
    return x2.max(dim=1).values

In [7]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

class ModifiedMobileNetV2Encoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # 1. 加载预训练的 MobileNetV2
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        self.backbone = mobilenet_v2(weights=weights).features
        
        # 2. 修改第一层卷积以接受 4 通道 (RGBA)
        old_conv = self.backbone[0][0]  # MobileNetV2 的第一层 Conv2d
        new_conv = nn.Conv2d(
            in_channels=4,               # 改为 4 通道
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None
        )
        
        # 3. 权重继承与 Alpha 通道初始化
        if pretrained:
            with torch.no_grad():
                # 复制前 3 个通道 (RGB) 的预训练权重
                new_conv.weight[:, :3, :, :] = old_conv.weight.clone()
                # 将第 4 个通道 (Alpha) 初始化为 0
                # 这样初始状态下 Alpha 通道不产生影响，随后可通过训练学习
                new_conv.weight[:, 3, :, :] = 0.0
                if old_conv.bias is not None:
                    new_conv.bias = old_conv.bias.clone()
                    
        # 替换原有的第一层
        self.backbone[0][0] = new_conv
        
        # 4. 冻结骨干网络 (保留第一层可训练)
        for name, param in self.backbone.named_parameters():
            # '0.0.weight' 和 '0.0.bias' 属于第一层卷积
            if not name.startswith('0.0.'):
                param.requires_grad = False
                
        # 5. 全局平均池化，将 2D 特征图转为 1D 向量
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
    def forward(self, x):
        # x shape: [B*N, 4, H, W]
        features = self.backbone(x)         # 输出 [B*N, 1280, H', W']
        pooled = self.global_pool(features) # 输出 [B*N, 1280, 1, 1]
        return pooled.view(pooled.size(0), -1) # 展平为 [B*N, 1280]


class PuzzlePoseModel(nn.Module):
    def __init__(self, pool_type="max_mean"):
        super().__init__()
        self.pool = pool_type
        
        # 骨干网络：输出特征维度固定为 1280
        self.encoder = ModifiedMobileNetV2Encoder(pretrained=True)
        d_model = 1280 
        
        # 计算全局上下文特征的维度
        ctx_dim = d_model * 2 if pool_type == "max_mean" else d_model
        
        # MLP 预测头的输入维度 = 局部特征(1280) + 全局特征(ctx_dim)
        in_dim = d_model * 2 + ctx_dim
        
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 3) # 输出预测的相对 dx, dy, drot
        )

    def forward(self, imgs, pad_mask):
        """
        imgs: [B, N, 4, H, W] - RGBA 图片张量
        pad_mask: [B, N] - 布尔张量，True 表示有效的拼图碎片
        returns: [B, N, N, 3] - 每对碎片(i, j)之间的相对位置和角度
        """
        B, N, C, H, W = imgs.shape
        
        # 将 Batch 和 N 维度合并以送入 CNN
        x = imgs.view(B * N, C, H, W)
        
        # 提取特征
        f = self.encoder(x)          # [B*N, 1280]
        f = f.view(B, N, -1)         # 恢复维度: [B, N, 1280]
        
        valid = ~pad_mask                         # [B,N] True where VALID

        if self.pool == "max":
            g = masked_max(f, valid)              # [B,D]
        elif self.pool == "mean":
            g = masked_mean(f, valid)             # [B,D]
        else:  # "maxmean"
            g = torch.cat([masked_max(f, valid), masked_mean(f, valid)], dim=-1)  # [B,2D]

        # 将全局特征扩展至 [B, N, N, ctx_dim]
        g_rep = g.unsqueeze(1).unsqueeze(2).expand(B, N, N, g.shape[-1]) 
        
        # --- 【修改点 2】: 构造两两相对特征矩阵 ---
        # f_i (Source碎片特征): [B, N, 1, 1280] -> [B, N, N, 1280]
        f_i = f.unsqueeze(2).expand(B, N, N, -1)
        # f_j (Target碎片特征): [B, 1, N, 1280] -> [B, N, N, 1280]
        f_j = f.unsqueeze(1).expand(B, N, N, -1)
        
        # 拼接 碎片i特征 + 碎片j特征 + 拼图全局特征
        cat_f = torch.cat([f_i, f_j, g_rep], dim=-1)  # [B, N, N, 1280*2 + ctx_dim]
        
        # 预测相对坐标与旋转角 (dx, dy, drot)
        out = self.head(cat_f)                        # [B, N, N, 3]
        
        return out


## 4. Training Loop

Implement your training loop here if you are using learning-based components.

You should:
- Log losses and relevant metrics.
- Respect the **parameter** and **runtime** constraints.
- Save any trained weights you intend to load in your test notebook as `username-assembly-model.pth`.


In [8]:
import torch.optim as optim

model = PuzzlePoseModel().to(DEVICE)

# 分离参数组
conv_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'encoder.backbone' in name:
        conv_params.append(param)
    elif 'head' in name:
        head_params.append(param)
        
print(f"Captured Conv params: {len(conv_params)}")
print(f"Captured Head params: {len(head_params)}")

# 设置 AdamW 优化器和差分学习率
optimizer = optim.AdamW([
    {'params': conv_params, 'lr': 1e-4}, # 骨干网络中未冻结的层
    {'params': head_params, 'lr': 1e-3}  # 随机初始化的预测头
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, 10))

import torch.nn.functional as F

def compute_pairwise_masked_loss(pred, gt_xy, gt_rot, pad_mask, loss_weights={'xy': 1.0, 'rot': 0.2}):
    """
    pred: [B, N, N, 3] (预测的相对 dx, dy, drot)
    gt_xy: [B, N, 2] (真实的绝对 x, y)
    gt_rot: [B, N] (真实的绝对 rot)
    pad_mask: [B, N] (True 表示 Padding)
    """
    # 1. 计算真实的相对位置和角度矩阵
    # gt_rel_xy = 碎片i位置 - 碎片j位置 => 形状 [B, N, N, 2]
    gt_rel_xy = gt_xy.unsqueeze(1) - gt_xy.unsqueeze(2) 
    
    # gt_rel_rot = 碎片i角度 - 碎片j角度 => 形状 [B, N, N]
    gt_rel_rot = gt_rot.unsqueeze(1) - gt_rot.unsqueeze(2)
    
    # 2. 生成两两之间的有效掩码
    valid_mask = ~pad_mask # [B, N]
    # 只有当 i 和 j 都是真实有效碎片时，这一对的 loss 才计入统计 [B, N, N]
    pair_valid_mask = valid_mask.unsqueeze(1) & valid_mask.unsqueeze(2)
    
    # 3. 切分预测结果
    pred_xy = pred[..., :2]
    pred_rot = pred[..., 2]
    
    # 4. 提取有效碎片对 (展平为1D以过滤 Padding 带来的 NaN 或零损失)
    valid_pred_xy = pred_xy[pair_valid_mask]
    valid_gt_xy = gt_rel_xy[pair_valid_mask]
    
    if valid_pred_xy.numel() == 0: 
        return (pred.sum() * 0.0), torch.tensor(0.0), torch.tensor(0.0)
    
    valid_pred_rot = pred_rot[pair_valid_mask]
    valid_gt_rot = gt_rel_rot[pair_valid_mask]
    
    # 5. 计算有效碎片对的 Loss
    loss_xy = F.smooth_l1_loss(valid_pred_xy, valid_gt_xy, reduction='mean')
    loss_rot = F.smooth_l1_loss(valid_pred_rot, valid_gt_rot, reduction='mean')
    
    # 加权总损失
    total_loss = loss_weights['xy'] * loss_xy + loss_weights['rot'] * loss_rot
    return total_loss, loss_xy, loss_rot


Captured Conv params: 1
Captured Head params: 8


In [9]:
train_loader = make_unified_puzzle_dataloader_zip(
    zip_path=DATASET_ZIP,
    split="train",
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUMBER_WORKERS,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    return_optional_images=False,
    augment_mode=DATASET_MODE
)

epochs = 10
model.train()

for epoch in range(epochs):
    total_train_loss = 0.0

    n_steps = 0

    
    # For speed/ETA style logging
    start_t = time.time()
    last_log_t = start_t
    
    for batch_idx, batch_data in enumerate(train_loader):
        # 1. 获取数据并移至 GPU
        pack = batch_fragments_from_collate(batch_data, device=DEVICE)
        imgs = pack['imgs'].to(DEVICE)       # [B, N, 4, H, W]
        gt_xy = pack['xy'].to(DEVICE)     # [B, N, 2]
        gt_rot = pack['rot'].to(DEVICE)   # [B, N]
        pad_mask = pack['pad_mask'].to(DEVICE)      # [B, N]
        valid_mask = ~pad_mask  # True where real fragments exist
        
        if torch.isnan(gt_xy).any() or torch.isnan(gt_rot).any():
            print(f"警告：第 {batch_idx} 个 Batch 的真实标签包含 NaN！")
            continue # 跳过这个有毒的批次
            
        # 2. 清空梯度
        optimizer.zero_grad()
        
        # 3. 前向传播
        pred = model(imgs, pad_mask)                      # [B, N, 3]
        if torch.isnan(pred).any():
            print("前方高能：模型输出已经是 NaN 了！")
            print("输入 imgs 包含 NaN 吗？", torch.isnan(imgs).any().item())
            # 如果输出是 NaN 但输入不是，说明是模型里的池化或权重炸了 (嫌疑人 1或4)
            break
        # 4. 计算遮罩 Loss
        loss, l_xy, l_rot = compute_pairwise_masked_loss(pred, gt_xy, gt_rot, valid_mask)
        if torch.isnan(loss):
            print("前方高能：Loss 计算结果为 NaN！模型输出正常。")
            # 说明是 Loss 函数的错位或空张量导致的 (嫌疑人 2或3)
            break
        # 5. 反向传播
        loss.backward()
        
        # 6. 梯度裁剪 (防止梯度爆炸，对拼接类任务很有用)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 7. 更新参数
        optimizer.step()
        
        total_train_loss += loss.item()
        
        n_steps += 1
        # ----------------------------
        # Iteration progress printing
        # ----------------------------
        log_every = 50
        if log_every and (batch_idx % log_every == 0):
            log_prefix = "train"
            
            now = time.time()
            dt = now - last_log_t
            elapsed = now - start_t
            steps_per_s = (log_every / dt) if dt > 0 else float("inf")

            # Running averages
            avg_loss = total_train_loss / n_steps

            # LR (for AdamW there is usually 1 param group)
            lr_val = None
            if optimizer is not None and optimizer.param_groups:
                lr_val = optimizer.param_groups[0].get("lr", None)
            
            total_steps = len(train_loader)
            if total_steps is not None:
                prog = f"{batch_idx:04d}/{total_steps:04d}"
            else:
                prog = f"{batch_idx:04d}/????"

            e_str = f"{epoch:03d}/{epochs:03d}" if (epoch and epochs) else ""

            lr_str = f" | lr {lr_val:.2e}" if lr_val is not None else ""
            prefix = f"{log_prefix} " if log_prefix else ""
            epoch_str = f"Epoch {e_str} | " if e_str else ""

            print(
                f"{prefix}{epoch_str}iter {prog}"
                f"{lr_str} | "
                f"{steps_per_s:.2f} it/s | {elapsed:.1f}s elapsed"
            )

            last_log_t = now
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_train_loss/len(train_loader):.4f}")
    scheduler.step()    


train iter 0000/1250 | lr 1.00e-04 | 18.35 it/s | 2.7s elapsed
train iter 0050/1250 | lr 1.00e-04 | 9.99 it/s | 7.7s elapsed
train iter 0100/1250 | lr 1.00e-04 | 10.85 it/s | 12.3s elapsed
train iter 0150/1250 | lr 1.00e-04 | 11.73 it/s | 16.6s elapsed
train iter 0200/1250 | lr 1.00e-04 | 11.70 it/s | 20.9s elapsed
train iter 0250/1250 | lr 1.00e-04 | 10.86 it/s | 25.5s elapsed
train iter 0300/1250 | lr 1.00e-04 | 11.67 it/s | 29.8s elapsed
train iter 0350/1250 | lr 1.00e-04 | 11.48 it/s | 34.1s elapsed
train iter 0400/1250 | lr 1.00e-04 | 11.78 it/s | 38.4s elapsed
train iter 0450/1250 | lr 1.00e-04 | 11.06 it/s | 42.9s elapsed
train iter 0500/1250 | lr 1.00e-04 | 11.59 it/s | 47.2s elapsed
train iter 0550/1250 | lr 1.00e-04 | 12.25 it/s | 51.3s elapsed
train iter 0600/1250 | lr 1.00e-04 | 12.56 it/s | 55.3s elapsed
train iter 0650/1250 | lr 1.00e-04 | 12.21 it/s | 59.4s elapsed
train iter 0700/1250 | lr 1.00e-04 | 12.13 it/s | 63.5s elapsed
train iter 0750/1250 | lr 1.00e-04 | 12.25 

## 5. Save Model (best)

In [10]:
def save_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    epoch: Optional[int] = None,
    best_val: Optional[float] = None,
    extra: Optional[Dict[str, Any]] = None,
):
    """
    Save a training checkpoint.

    Args:
        path: output .pt or .pth file
        model: model to save
        optimizer: optimizer (optional)
        scheduler: LR scheduler (optional)
        epoch: current epoch (optional)
        best_val: best validation loss so far (optional)
        extra: any additional metadata to store
    """

    ckpt = {
        "model_state": model.state_dict(),
    }

    if optimizer is not None:
        ckpt["optimizer_state"] = optimizer.state_dict()
    if scheduler is not None:
        ckpt["scheduler_state"] = scheduler.state_dict()
    if epoch is not None:
        ckpt["epoch"] = epoch
    if best_val is not None:
        ckpt["best_val"] = best_val
    if extra is not None:
        ckpt["extra"] = extra

    torch.save(ckpt, path)
    print(f"Checkpoint saved to: {path}")


save_checkpoint(
    path=OUTPUT_MODEL,
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=EPOCHS,
    best_val=best_val,
)

NameError: name 'best_val' is not defined

## Qualitative Visualisation

In [ ]:
import numpy as np
from PIL import Image

def assemble_canvas(
    imgs_norm,
    xy,
    rot_deg,
    pad_mask,
    img_sizes,
    canvas_size=(240, 240),
    background=(255, 255, 255, 255),
    assume_xy_is_center=True,
):
    """
    Assemble fragments into a canvas in the same coordinate system as xy.

    imgs_norm: [N,C,H,W] or [B,N,C,H,W] in [0,1] (after denormalize)
    xy:        [N,2] or [B,N,2] in reference pixels
    img_sizes  [N,2] original fragment sizes
    rot_deg:   [N] or [B,N]
    pad_mask:  [N] or [B,N]
    pad_mask:  [N,2] or [B,N,2]
    """

    # ---- tensors -> numpy ----
    imgs = imgs_norm.detach().cpu().numpy()
    xy = xy.detach().cpu().numpy()
    rot_deg = rot_deg.detach().cpu().numpy()
    pad_mask = pad_mask.detach().cpu().numpy()
    img_sizes = img_sizes.detach().cpu().numpy()

    # ---- unwrap batch if present ----
    if imgs.ndim == 5: imgs = imgs[0]
    if xy.ndim == 3: xy = xy[0]
    if rot_deg.ndim == 2: rot_deg = rot_deg[0]
    if pad_mask.ndim == 2: pad_mask = pad_mask[0]

    N = imgs.shape[0]

    # ----------------------------
    # Convert images to PIL RGBA (and upscale to target_tile_size)
    # ----------------------------
    print("start Convert images to PIL RGBA")
    pil_imgs = []
    for i in range(N):
        img = imgs[i]  # [C,H,W] likely

        # CHW -> HWC
        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.transpose(img, (1, 2, 0))

        # Ensure [0,1]
        if img.min() < 0:
            img = (img + 1) / 2

        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        # Ensure RGBA
        if img.shape[-1] == 3:
            alpha = np.full((*img.shape[:2], 1), 255, dtype=np.uint8)
            img = np.concatenate([img, alpha], axis=-1)
        elif img.shape[-1] == 4:
            pass
        else:
            raise ValueError(f"Expected 3 or 4 channels after conversion, got {img.shape}")

        pil = Image.fromarray(img, mode="RGBA")

        img_size = img_sizes[i]

        # Upscale to original size
        if img_size is not None:
            pil = pil.resize((img_size[0], img_size[1]), resample=Image.NEAREST)

        pil_imgs.append(pil)

    # ----------------------------
    # Create canvas
    # ----------------------------
    W, H = canvas_size
    canvas = Image.new("RGBA", (W, H), background)

    # ----------------------------
    # Composite fragments
    # ----------------------------
    print("start Composite fragments")
    for i in range(N):
        if pad_mask[i]:
            continue

        img = pil_imgs[i]
        x, y = float(xy[i, 0]) * W, float(xy[i, 1]) * H
        theta = float(rot_deg[i])

        w, h = img.size
        cx, cy = w / 2, h / 2

        # Rotate around centre
        rot_img = img.rotate(-theta, resample=Image.BICUBIC, expand=True)
        rw, rh = rot_img.size
        dx = rw / 2 - cx
        dy = rh / 2 - cy

        # If xy is centre coords, convert to top-left for pasting
        if assume_xy_is_center:
            paste_x = int(round(x - cx - dx)) 
            paste_y = int(round(y - cy - dy)) 
        else:
            paste_x = int(round(x - dx)) 
            paste_y = int(round(y - dy)) 

        canvas.alpha_composite(rot_img, (paste_x, paste_y))

    return canvas

# ----------------------------
# Reload training dataloader too
# ----------------------------
train_loader = make_unified_puzzle_dataloader_zip(
    zip_path=DATASET_ZIP,
    split="train",
    batch_size=1,
    shuffle=False,
    num_workers=0,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    return_optional_images=False,
    augment_mode="None"
)
val_loader = make_unified_puzzle_dataloader_zip(
    zip_path=DATASET_ZIP,
    split="val",
    batch_size=1,
    shuffle=False,
    num_workers=0,
    fixed_size=FRAGMENT_FIXED_SIZE,
    normalize_rgb=NORMALIZE_RGB,
    return_optional_images=False,
    augment_mode="None"
)

NUM_TRAIN_EXAMPLES = 1
NUM_VAL_EXAMPLES = 1

# --------------------------------
# RGB de-normalisation (ImageNet-style)
# --------------------------------
RGB_MEAN = torch.tensor([0.485, 0.456, 0.406])
RGB_STD  = torch.tensor([0.229, 0.224, 0.225])

def denormalize_imgs(imgs: torch.Tensor) -> torch.Tensor:
    """
    imgs: [B,N,C,H,W] where C is 3 (RGB) or 4 (RGBA)
          If C==4, assumes channel 3 is alpha and leaves it unchanged.
    returns: same shape, RGB in [0,1] (alpha preserved if present)
    """
    assert imgs.ndim == 5, f"Expected [B,N,C,H,W], got {tuple(imgs.shape)}"
    B, N, C, H, W = imgs.shape
    if C not in (3, 4):
        raise ValueError(f"Expected C=3 or C=4, got C={C}")

    mean = RGB_MEAN.view(1, 1, 3, 1, 1).to(device=imgs.device, dtype=imgs.dtype)
    std  = RGB_STD.view(1, 1, 3, 1, 1).to(device=imgs.device, dtype=imgs.dtype)

    out = imgs.clone()
    out[..., :3, :, :] = (out[..., :3, :, :] * std + mean).clamp(0.0, 1.0)

    # If alpha exists, you typically want it in [0,1] for display as well
    if C == 4:
        out[..., 3:4, :, :] = out[..., 3:4, :, :].clamp(0.0, 1.0)

    return out

def render_examples(loader, split_name: str, num_examples: int):
    shown = 0
    with torch.no_grad():
        for batch in loader:
            pack = batch_fragments_from_collate(batch, device=DEVICE)
            

            imgs = pack["imgs"]          # [1,N,C,H,W]
            gt_xy = pack["xy"]           # [1,N,2]
            gt_rot = pack["rot"]         # [1,N]
            pad_mask = pack["pad_mask"]  # [1,N]
            img_sizes = pack["img_sizes"]  # [1,N,2]

            pred = model(imgs, pad_mask=pad_mask)
            print("get model prediction")
            pred_xy = pred[..., :2]
            pred_rot = pred[..., 2]


            canvas_width = int(pack["meta"][0]["xy_canvas_W"])
            canvas_height = int(pack["meta"][0]["xy_canvas_H"])

            imgs_norm = denormalize_imgs(imgs) # Remove the normalisation that occurs in dataloader

            print("imgs_norm:", imgs_norm.shape)        # [B,N,C,H,W]
            print("gt_xy:", gt_xy.shape, gt_xy.dtype)   # [B,N,2]
            print("gt_xy min/max:", gt_xy[0].min(0).values, gt_xy[0].max(0).values)
            print("fragment size:", imgs_norm.shape[-2:], "(H,W)")
            print("start assemble canvas")
            gt_canvas = assemble_canvas(imgs_norm[0], gt_xy[0], gt_rot[0], pad_mask[0],img_sizes=img_sizes[0],canvas_size=(canvas_width,canvas_height))
            pr_canvas = assemble_canvas(imgs_norm[0], pred_xy[0], pred_rot[0], pad_mask[0],img_sizes=img_sizes[0],canvas_size=(canvas_width,canvas_height) )


            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(np.asarray(gt_canvas))
            axes[0].set_title(f"{split_name}: Ground Truth Assembly")
            axes[0].axis("off")

            axes[1].imshow(np.asarray(pr_canvas))
            axes[1].set_title(f"{split_name}: Predicted Assembly")
            axes[1].axis("off")
            plt.tight_layout()
            plt.show()

            shown += 1
            if shown >= num_examples:
                break

# ----------------------------
# Render training + validation
# ----------------------------
model.eval()
print("start sampling")
render_examples(train_loader, "TRAIN", NUM_TRAIN_EXAMPLES)
render_examples(val_loader, "VAL", NUM_VAL_EXAMPLES)

start sampling


## 6. Count Number of Parameters (Do not remove)
Before training, we compute how many trainable parameters the model has.

A parameter is a number the model learns during training (for example, weights in convolutional layers or attention layers).
Only parameters with requires_grad = True are updated by the optimizer. Some parameters may be frozen (not trained), and those are intentionally excluded here.

This count is important because:
 - It tells you how large your model is.
 - Larger models are harder to train and easier to overfit.
 - When you change the architecture (e.g., d_model, number of layers), this number should change.

When you change the input resolution (e.g., 128 → 24), the number of parameters does not change, which is an important concept to understand.

⚠️ Important:
If you freeze or unfreeze parameters after this cell runs, the reported number will be incorrect.
This cell must run after the model is fully constructed and before training begins.

In [10]:
def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_trainable_parameters(model)
print(f"Trainable parameters: {num_params:,}")

# Hard safety check for oversized models
MAX_PARAMS = 15_000_000

if num_params > MAX_PARAMS:
    print("\n" + "!" * 80)
    print("WARNING: MODEL IS TOO LARGE")
    print(f"This model has {num_params:,} trainable parameters.")
    print(f"The recommended maximum for this assignment is {MAX_PARAMS:,}.")
    print()
    print("Large models:")
    print("- Train much more slowly")
    print("- Are more likely to overfit")
    print("- May exceed memory or runtime limits")
    print()
    print("Consider reducing:")
    print("- d_model")
    print("- number of Transformer layers")
    print("- number of attention heads")
    print("!" * 80 + "\n")

Trainable parameters: 259,683


## 7. Time Script (Do not remove)

In [11]:
script_end = time.perf_counter()
total_time = script_end - script_start

print(f"Total execution time: {total_time:.6f} seconds")

Total execution time: 2968.166979 seconds
